# Compare saved experiments
Select runs below: every comparison panel draws one curve per run and method, with consistent colors. Training history and CP/APS results are read from disk by default; no model or data cache is loaded. Optional reevaluation uses frozen checkpoints sequentially, without training.

Plots remain here for editing. t-SNE uses one row per run (task, domain, and data split) and the same sampled segments; axes from separate fits are not aligned.
New runs report validation CE/accuracy averaged equally across domains. Reevaluation restores the saved four-way split, including isolated validation domains; it never resplits or retrains.


In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timezone

project_root = None  # Set the project directory if launching elsewhere.
PROJECT_ROOT = Path(project_root or Path.cwd()).expanduser().resolve()
if (PROJECT_ROOT / "step02_models").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
PYTHON_ROOT = PROJECT_ROOT / "python"
if str(PYTHON_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTHON_ROOT))
from step07_evaluation.comparison import resolve_result, load_result, validate_comparison

for path in sorted((PYTHON_ROOT / "runs").rglob("experiment.json")):
    print(path.parent.relative_to(PYTHON_ROOT / "runs"))


## Selection
Paths are relative to `python/runs/` or absolute. A run path selects its latest completed `conformal-*`; an explicit conformal directory pins the evaluation. Remove/add entries to change the number of experiment curves. All saved CP/APS methods are displayed; color identifies the run and line style identifies the method.

Set `recompute=True` only to rerun inference/calibration; `evaluation_config_path` then applies the same evaluation override to every run. Existing saved evaluations are preserved.

After updating Python modules, restart the kernel and run from the top to avoid cached function definitions.

In [ ]:
selected_runs = {
    "baseline": "readme-ablation/01_plain",
    "DA": "readme-ablation/02_da",
    "CL": "readme-ablation/03_cl",
    "DA + CL": "readme-ablation/04_da_cl",
}
distribution_alpha = 0.1
show_nn_accuracy = True  # Optional horizontal references, in each run's color.
run_embedding_projections = True
viz_seed = 42
viz_max_per_split = 1000  # None keeps all segments.

recompute = False
checkpoint_name = "best.pt"  # "last.pt" evaluates the final epoch; requires recompute=True
evaluation_config_path = None
device_name = "auto"
log_to_wandb = False  # Used only when recompute=True.
comparison_output = None  # None creates python/comparisons/<timestamp>; or set an output directory.


In [ ]:
# Clear downstream state before loading; a failed selection must not reuse old curves.
results = {}
projection_data = coordinates = None
if not selected_runs:
    raise ValueError("Select at least one run")
if not 0 < distribution_alpha < 1:
    raise ValueError("distribution_alpha must be between 0 and 1")
if not recompute and (evaluation_config_path is not None or log_to_wandb):
    raise ValueError("Evaluation overrides and W&B logging require recompute=True")
resolved = {name: resolve_result(path, PYTHON_ROOT / "runs") for name, path in selected_runs.items()}
if len({run for run, saved in resolved.values()}) != len(resolved):
    raise ValueError("Select each run only once")
loaded = {}
for name, (run_dir, saved_dir) in resolved.items():
    if recompute:
        from step07_evaluation.restoration import reevaluate_run
        saved_dir = reevaluate_run(run_dir, PYTHON_ROOT / "dataset", saved_dir=saved_dir,
                                  override_path=evaluation_config_path, checkpoint_name=checkpoint_name,
                                  device_name=device_name, log_to_wandb=log_to_wandb)
    if saved_dir is None:
        raise FileNotFoundError(f"{name}: no completed evaluation; set recompute=True")
    loaded[name] = load_result(saved_dir)
validate_comparison(loaded)
results = loaded
output_dir = (Path(comparison_output).expanduser().resolve() if comparison_output is not None
              else PYTHON_ROOT / "comparisons" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
output_dir.mkdir(parents=True, exist_ok=True)
colors = dict(zip(results, plt.get_cmap("tab10")(np.linspace(0, 1, len(results)))))
(output_dir / "selection.json").write_text(json.dumps({
    "runs": {name: str(result["directory"]) for name, result in results.items()},
    "methods": {name: list(result["method_results"]) for name, result in results.items()}, "distribution_alpha": distribution_alpha,
    "viz_seed": viz_seed, "viz_max_per_split": viz_max_per_split,
}, indent=2))
for name, result in results.items():
    history = result["history"]
    print(f"{name}: epochs={len(history['history'])}, best={history.get('best_epoch')}, "
          f"NN accuracy={result['nn_accuracy']:.3f} | {result['directory']}")
print("Comparison outputs:", output_dir)


## Training comparison
Each panel contains one line per run. Loss is task classification CE (the total training objective differs when DA/CL are enabled). Different stopping epochs keep their actual lengths; dots mark selected checkpoints.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7), squeeze=False)
for column, metric in enumerate(("loss", "accuracy")):
    for row, split in enumerate(("train", "validation")):
        ax = axes[row, column]
        for name, result in results.items():
            history = result["history"]["history"]
            epochs = [r["epoch"] for r in history]
            values = [r[f"{split}_{metric}"] for r in history]
            ax.plot(epochs, values, color=colors[name], label=name)
            best = result["history"].get("best_epoch")
            if best in epochs:
                ax.scatter(best, values[epochs.index(best)], color=colors[name], s=30)
        ax.set(xlabel="Epoch", ylabel=f"{split.capitalize()} {metric}")
        if metric == "accuracy":
            ax.set_ylim(0, 1.02)
        ax.legend()
fig.tight_layout()
fig.savefig(output_dir / "training_comparison.png", dpi=180)
plt.show()
plt.close(fig)


## Coverage and set size

In [ ]:
# Colors/markers identify experimental methods; line styles identify prediction models.
from matplotlib.lines import Line2D
from itertools import cycle

markers = dict(zip(results, cycle(("o", "s", "^", "D", "v", "P", "X", "*"))))
methods = list(dict.fromkeys(method for result in results.values() for method in result["method_results"]))
method_styles = {method: ("--" if method == "APS unweighted" else ":" if method == "APS weighted" else "-")
                 for method in methods}
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for name, result in results.items():
    for method, rows in result["method_results"].items():
        levels = [r["alpha"] for r in rows]
        for ax, metric in zip(axes, ("coverage", "mean_set_size")):
            ax.plot(levels, [r[metric] for r in rows], marker=markers[name], linestyle=method_styles[method],
                    color=colors[name], label=f"{name} | {method}")
    if show_nn_accuracy:
        axes[0].axhline(result["nn_accuracy"], color=colors[name], linestyle="-.", alpha=0.65,
                       label=f"{name} NN = {result['nn_accuracy']:.3f}")
levels = sorted({r["alpha"] for result in results.values() for rows in result["method_results"].values() for r in rows})
axes[0].plot(levels, 1 - np.asarray(levels), "--", color="gray")
nominal_x = levels[len(levels) // 2]
axes[0].annotate("Nominal 1-alpha", xy=(nominal_x, 1 - nominal_x),
                 xytext=(0.65, 0.83), textcoords="axes fraction", ha="center", fontsize=9,
                 arrowprops={"arrowstyle": "->", "color": plt.rcParams["text.color"]})
axes[0].set(xlabel="Alpha", ylabel="Coverage / accuracy", ylim=(0, 1.02))
axes[1].set(xlabel="Alpha", ylabel="Mean prediction-set size", ylim=(0, len(next(iter(results.values()))["class_names"])))
# Two shared legend boxes avoid repeating every run/model combination.
run_handles = [Line2D([], [], color=colors[name], marker=markers[name], linestyle="None", label=name)
               for name in results]
model_handles = [Line2D([], [], color=plt.rcParams["text.color"], linestyle=method_styles[method], label=method)
                 for method in methods]
if show_nn_accuracy:
    model_handles.append(Line2D([], [], color=plt.rcParams["text.color"], linestyle="-.", label="NN accuracy"))
fig.legend(handles=run_handles, title="Method", loc="lower center", bbox_to_anchor=(0.27, 0.01),
           ncol=min(4, len(run_handles)), fontsize=9)
fig.legend(handles=model_handles, title="Model", loc="lower center", bbox_to_anchor=(0.75, 0.01),
           ncol=min(3, len(model_handles)), fontsize=9)
fig.suptitle("CP and randomized APS comparison")
fig.tight_layout(rect=(0, 0.16, 1, 0.95))
fig.savefig(output_dir / "coverage_set_size.png", dpi=180)
plt.show()
plt.close(fig)


## Prediction-set cardinality distribution
Computed from saved p-values at the same alpha; includes empty sets.

In [ ]:
# Integer-centered bins; each method gets a panel with grouped bars for the selected runs.
class_count = len(next(iter(results.values()))["class_names"])
cardinalities = np.arange(class_count + 1)
bin_edges = np.arange(class_count + 2) - 0.5
histogram_methods = list(dict.fromkeys(method for result in results.values()
                                     for method in result["method_p_values"]))
fig, axes = plt.subplots(1, len(histogram_methods), figsize=(6 * len(histogram_methods), 4.5),
                         sharey=True, squeeze=False)
for ax, method in zip(axes.flat, histogram_methods):
    names = [name for name, result in results.items() if method in result["method_p_values"]]
    sizes = [(results[name]["method_p_values"][method] > distribution_alpha).sum(axis=1)
             for name in names]
    # Weights normalize each run separately: bar heights sum to one per run.
    ax.hist(sizes, bins=bin_edges, weights=[np.ones(len(values)) / len(values) for values in sizes],
            histtype="bar", stacked=False, rwidth=0.85,
            color=[colors[name] for name in names], label=names)
    ax.set(xlabel="Prediction-set cardinality", xticks=cardinalities,
           xlim=(-0.5, class_count + 0.5), ylim=(0, 1.02), title=method)
    ax.legend(fontsize=8)
axes[0, 0].set_ylabel("Fraction of test samples")
fig.suptitle(f"Prediction-set size distribution | alpha={distribution_alpha:g}")
fig.tight_layout()
fig.savefig(output_dir / "set_size_distribution.png", dpi=180)
plt.show()
plt.close(fig)


## t-SNE computation
Fit once per run; rerun only the next plotting cell to adjust style or layout. Sampling, seed, and class colors are shared. Different learned embeddings are projected independently.

In [ ]:
from step07_evaluation.projections import load_comparison_projections
from sklearn.manifold import TSNE

projection_data = coordinates = None
if run_embedding_projections:
    loaded = load_comparison_projections(
        {name: result["directory"] for name, result in results.items()}, viz_seed, viz_max_per_split)
    computed = {}
    for index, (name, data) in enumerate(loaded.items(), start=1):
        embeddings = data["embeddings"]
        if min(embeddings.shape) < 2:
            raise ValueError(f"{name}: t-SNE PCA initialization needs at least two samples and features")
        mapper = TSNE(n_components=2, perplexity=min(30.0, len(embeddings) - 1),
                      init="pca", learning_rate="auto", random_state=viz_seed)
        print(f"Computing t-SNE: {name}, n={len(embeddings)}", flush=True)
        computed[name] = mapper.fit_transform(embeddings)
        np.savez_compressed(output_dir / f"tsne_{index:02d}.npz", name=name, coordinates=computed[name],
                            **{key: data[key] for key in ("recording", "segment_index", "labels", "domains", "splits")},
                            parameters=json.dumps(mapper.get_params()), seed=viz_seed)
    projection_data, coordinates = loaded, computed


In [ ]:
from matplotlib.ticker import MaxNLocator

# Each run occupies a row; all three columns reuse its cached coordinates.
if coordinates is not None:
    fields = ("labels", "domains", "splits")
    categories = {field: list(dict.fromkeys(value for data in projection_data.values()
                                           for value in data[field])) for field in fields}
    # Derive OOD labels from the saved split, not a user's position in the legend.
    held_out_users = set(next(iter(projection_data.values()))["held_out_domains"])
    source_users = [name for name in categories["domains"] if name not in held_out_users]
    target_users = [name for name in categories["domains"] if name in held_out_users]
    user_aliases = {name: f"user_{i + 1}" for i, name in enumerate(source_users)}
    user_aliases.update({name: "OOD user" if len(target_users) == 1 else f"OOD user_{i + 1}"
                         for i, name in enumerate(target_users)})
    palettes = {field: plt.get_cmap("tab20", len(categories[field])) for field in fields}
    fig, axes = plt.subplots(len(coordinates), 3, figsize=(16, 4.5 * len(coordinates)),
                             sharex="row", sharey="row", squeeze=False)
    for row, (name, points) in enumerate(coordinates.items()):
        data = projection_data[name]
        titles = (f"Task: {data['task']}", f"Domain: {data['domain_key']}", "Data split")
        for ax, field, title in zip(axes[row], fields, titles):
            for i, category in enumerate(categories[field]):
                selected = data[field] == category
                ax.scatter(*points[selected].T, s=12, alpha=0.65,
                           color=palettes[field](i),
                           label=user_aliases[category] if field == "domains" and data["domain_key"] == "user" else category,
                           linewidths=0)
            ax.set(title=title, xlabel="t-SNE 1")
            ax.xaxis.set_major_locator(MaxNLocator(4))
            ax.legend(fontsize=8, markerscale=1.5)
        axes[row, 0].set_ylabel(f"{name} | n={len(points)}\nt-SNE 2")
    fig.suptitle("t-SNE comparison")
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    fig.savefig(output_dir / "tsne_comparison.png", dpi=180)
    plt.show()
    plt.close(fig)
